# MaritimeBERT Domain-Adaptive Pretraining (DAPT) - Google Colab Runner

This notebook executes the **ModernBERT DAPT Pipeline** (`MaritimeBERT-v1`) entirely on Google Colab's cloud GPU hardware.

---

## Step 1: Ensure Files are Available to Colab

### Option A: Mount Google Drive (Recommended for persistent checkpoints & metrics)
Run Cell 1 below to mount Google Drive and navigate to your uploaded project directory.

### Option B: Direct GitHub / Upload
If you uploaded or cloned the project directly into `/content/TSBC-Pipeline`, navigate into your repository root directory.

---

## Step 2: Set Colab Runtime to GPU
1. Go to **Runtime** > **Change runtime type**.
2. Under **Hardware accelerator**, select **T4 GPU** (or A100/V100 if available).
3. Click **Save**.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
# Cell 1: Mount Google Drive & Set Working Directory
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')

    # Adjust this path if your project directory name in Drive is different
    drive_project_path = '/content/drive/MyDrive/Abhishek-TSBC/TSBC-Pipeline'
    if os.path.exists(drive_project_path):
        os.chdir(drive_project_path)
        print("Successfully changed working directory to:", os.getcwd())
    else:
        print("Current working directory:", os.getcwd())
except Exception as e:
    print("Drive mount skipped or not running in Colab:", e)

Mounted at /content/drive
Successfully changed working directory to: /content/drive/MyDrive/Abhishek-TSBC/TSBC-Pipeline


## 1. Verify GPU Hardware Runtime

In [2]:
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("GPU Memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
!nvidia-smi

PyTorch Version: 2.11.0+cu128
CUDA Available: True
GPU Device Name: Tesla T4
GPU Memory (GB): 15.64
Wed Sep 16 16:16:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                       

## 2. Install DAPT Pipeline Requirements

In [3]:
!pip install torch transformers accelerate datasets pyyaml pandas scipy

## 3. Step 1: Corpus Inspection & Cryptographic SHA-256 Hash Lock

In [4]:
!python dapt/scripts/inspect_corpus.py --config dapt/configs/dapt.yaml

[2026-09-16 16:16:59] [INFO] [dapt] Analyzing corpus at: /content/drive/MyDrive/Abhishek-TSBC/TSBC-Pipeline/outputs/maritime_corpus.txt
[2026-09-16 16:17:01] [INFO] [dapt] Corpus manifest saved to: /content/drive/MyDrive/Abhishek-TSBC/TSBC-Pipeline/dapt/outputs/data/corpus_manifest.json
[2026-09-16 16:17:01] [INFO] [dapt] === Corpus Inspection Summary ===
[2026-09-16 16:17:01] [INFO] [dapt] Corpus File: maritime_corpus.txt
[2026-09-16 16:17:01] [INFO] [dapt] Absolute Corpus Path: /content/drive/MyDrive/Abhishek-TSBC/TSBC-Pipeline/outputs/maritime_corpus.txt
[2026-09-16 16:17:01] [INFO] [dapt] SHA-256: 852fea9a1d756a6989fd55f40646fd7e92ef7597baa4ada36d450e7e8acb6e11
[2026-09-16 16:17:01] [INFO] [dapt] File Size: 24,556,236 bytes
[2026-09-16 16:17:01] [INFO] [dapt] Total Documents: 96,861
[2026-09-16 16:17:01] [INFO] [dapt] Total Words: 3,830,350
[2026-09-16 16:17:01] [INFO] [dapt] Total Characters: 24,356,820
[2026-09-16 16:17:01] [INFO] [dapt] Unique Vocabulary: 80,333
[2026-09-16 16:1

## 4. Step 2: Prepare Reproducible Dataset Splits & Leakage Diagnostics

In [5]:
!python dapt/scripts/prepare_dataset.py --config dapt/configs/dapt.yaml

[2026-09-16 16:18:08] [INFO] [dapt] Split dataset (96861 total docs) -> Train: 87174, Val: 4843, Test: 4844
[2026-09-16 16:18:09] [INFO] [dapt] Saved dataset splits and split_manifest.json to /content/drive/MyDrive/Abhishek-TSBC/TSBC-Pipeline/dapt/outputs/data
[2026-09-16 16:18:09] [INFO] [dapt] === Dataset Preparation Summary ===
[2026-09-16 16:18:09] [INFO] [dapt] Train Documents: 87,174
[2026-09-16 16:18:09] [INFO] [dapt] Val Documents: 4,843
[2026-09-16 16:18:09] [INFO] [dapt] Test Documents: 4,844
[2026-09-16 16:18:09] [INFO] [dapt] Exact Leakage (Train/Val): 0
[2026-09-16 16:18:09] [INFO] [dapt] Exact Leakage (Train/Test): 0
[2026-09-16 16:18:09] [INFO] [dapt] Split files written to: /content/drive/MyDrive/Abhishek-TSBC/TSBC-Pipeline/dapt/outputs/data


## 5. Step 3: ModernBERT Tokenizer Diagnostics

In [6]:
!python dapt/scripts/tokenize_corpus.py --config dapt/configs/dapt.yaml

[2026-09-16 16:21:51] [INFO] [dapt] Loading tokenizer from: answerdotai/ModernBERT-base
config.json: 100% 1.19k/1.19k [00:00<00:00, 3.59MB/s]
tokenizer_config.json: 100% 20.8k/20.8k [00:00<00:00, 59.4MB/s]
tokenizer.json: 100% 2.13M/2.13M [00:00<00:00, 58.1MB/s]
special_tokens_map.json: 100% 694/694 [00:00<00:00, 3.33MB/s]
[2026-09-16 16:21:53] [INFO] [dapt] Resolved document boundary token: sep_token ([SEP]) (ID: 50282)
[2026-09-16 16:21:53] [INFO] [dapt] Analyzing tokenizer metrics on 96861 documents (sample size cap: 10000)...
[2026-09-16 16:21:55] [INFO] [dapt] Tokenizer report saved to: /content/drive/MyDrive/Abhishek-TSBC/TSBC-Pipeline/dapt/outputs/data/tokenizer_report.json
[2026-09-16 16:21:55] [INFO] [dapt] === Tokenizer Diagnostic Summary ===
[2026-09-16 16:21:55] [INFO] [dapt] Model/Tokenizer: answerdotai/ModernBERT-base
[2026-09-16 16:21:55] [INFO] [dapt] Vocab Size: 50,368
[2026-09-16 16:21:55] [INFO] [dapt] Resolved Boundary Token: sep_token ([SEP])
[2026-09-16 16:21:55] 

## 6. Step 4: Validate Dataset & Document Sequence Packing Efficiency

In [7]:
!python dapt/scripts/validate_dataset.py --config dapt/configs/dapt.yaml

[2026-09-16 16:22:59] [INFO] [dapt] Packing 1000 documents into max_seq_length=512 blocks...
[2026-09-16 16:22:59] [INFO] [dapt] Packing complete: 1000 docs -> 120 sequences. Efficiency: 99.99% (Padding waste: 0.01%)
[2026-09-16 16:22:59] [INFO] [dapt] === Dataset & Packing Validation Summary ===
[2026-09-16 16:22:59] [INFO] [dapt] Sample Documents Evaluated: 1,000
[2026-09-16 16:22:59] [INFO] [dapt] Max Sequence Length: 512
[2026-09-16 16:22:59] [INFO] [dapt] Packed Sequence Count: 120
[2026-09-16 16:22:59] [INFO] [dapt] Packing Efficiency: 99.99%
[2026-09-16 16:22:59] [INFO] [dapt] Padding Waste: 0.01%
[2026-09-16 16:22:59] [INFO] [dapt] Dataset validation passed successfully!


## 7. Step 5: Evaluate Untouched ModernBERT Baseline on Held-Out Set

In [8]:
!python dapt/scripts/evaluate_dapt.py --config dapt/configs/dapt.yaml --is-baseline --eval-split validation

[2026-09-16 16:23:22] [INFO] [dapt] Evaluating untouched baseline model: answerdotai/ModernBERT-base
[2026-09-16 16:23:22] [INFO] [dapt] Split dataset (96861 total docs) -> Train: 87174, Val: 4843, Test: 4844
[2026-09-16 16:23:23] [INFO] [dapt] Packing 4843 documents into max_seq_length=512 blocks...
[2026-09-16 16:23:24] [INFO] [dapt] Packing complete: 4843 docs -> 585 sequences. Efficiency: 99.89% (Padding waste: 0.11%)

model.safetensors: downloading bytes:  55% 331M/599M [00:01<00:00, 355MB/s, 28.2MB/s  ]
model.safetensors: reconstructing file:  22% 134M/599M [00:01<00:05, 88.8MB/s]
model.safetensors: downloading bytes:  70% 419M/599M [00:01<00:00, 281MB/s, 34.8MB/s  ]
model.safetensors: downloading bytes:  92% 553M/599M [00:02<00:00, 322MB/s, 46.0MB/s  ]
model.safetensors: reconstructing file:  67% 402M/599M [00:02<00:00, 197MB/s, 31.1MB/s  ]
model.safetensors: reconstructing file:  78% 469M/599M [00:02<00:00, 204MB/s, 36.5MB/s  ]
model.safetensors: downloading bytes: 100% 569M/56

## 8. Step 6: Execute ModernBERT Domain-Adaptive Pretraining (MaritimeBERT-v1) on GPU

In [9]:
!python dapt/scripts/train_dapt.py --config dapt/configs/dapt.yaml

[2026-09-16 16:27:08] [INFO] [dapt] System Environment -> Torch: 2.11.0+cu128, CUDA: True
[2026-09-16 16:27:08] [INFO] [dapt] Analyzing corpus at: /content/drive/MyDrive/Abhishek-TSBC/TSBC-Pipeline/outputs/maritime_corpus.txt
[2026-09-16 16:27:09] [INFO] [dapt] Split dataset (96861 total docs) -> Train: 87174, Val: 4843, Test: 4844
[2026-09-16 16:27:10] [INFO] [dapt] Packing 87174 documents into max_seq_length=512 blocks...
[2026-09-16 16:27:27] [INFO] [dapt] Packing complete: 87174 docs -> 10484 sequences. Efficiency: 99.99% (Padding waste: 0.01%)
[2026-09-16 16:27:27] [INFO] [dapt] Packing 4843 documents into max_seq_length=512 blocks...
[2026-09-16 16:27:28] [INFO] [dapt] Packing complete: 4843 docs -> 585 sequences. Efficiency: 99.89% (Padding waste: 0.11%)
[2026-09-16 16:27:28] [INFO] [dapt] Loading ModernBERT MLM model from: answerdotai/ModernBERT-base
Loading weights: 100% 137/137 [00:00<00:00, 5722.85it/s]
[2026-09-16 16:27:29] [INFO] [dapt] Automatically placed model on CUDA G

## 9. Step 7: Evaluate Trained MaritimeBERT-v1 Model

In [10]:
!python dapt/scripts/evaluate_dapt.py --config dapt/configs/dapt.yaml --eval-split validation

[2026-09-16 18:19:09] [INFO] [dapt] Evaluating final MaritimeBERT-v1 model: /content/drive/MyDrive/Abhishek-TSBC/TSBC-Pipeline/dapt/outputs/experiments/MaritimeBERT-v1
[2026-09-16 18:19:09] [INFO] [dapt] Split dataset (96861 total docs) -> Train: 87174, Val: 4843, Test: 4844
[2026-09-16 18:19:10] [INFO] [dapt] Packing 4843 documents into max_seq_length=512 blocks...
[2026-09-16 18:19:11] [INFO] [dapt] Packing complete: 4843 docs -> 585 sequences. Efficiency: 99.89% (Padding waste: 0.11%)
Loading weights: 100% 137/137 [00:00<00:00, 1839.29it/s]
[2026-09-16 18:19:13] [INFO] [dapt] Evaluating model on 585 packed evaluation samples...
[2026-09-16 18:19:49] [INFO] [dapt] Evaluation complete -> MLM Loss: 0.5247, Perplexity: 1.69
[2026-09-16 18:19:49] [INFO] [dapt] Evaluation complete. Metrics saved to: /content/drive/MyDrive/Abhishek-TSBC/TSBC-Pipeline/dapt/outputs/experiments/MaritimeBERT-v1/evaluation_metrics.json


## 10. Step 8: Compare Baseline ModernBERT vs. MaritimeBERT-v1 Metrics

In [11]:
!python dapt/scripts/compare_runs.py

[2026-09-16 18:20:10] [INFO] [dapt] === Baseline vs MaritimeBERT-v1 Evaluation Comparison ===
[2026-09-16 18:20:10] [INFO] [dapt] Evaluation Split: validation
[2026-09-16 18:20:10] [INFO] [dapt] Baseline MLM Loss:   1.5365
[2026-09-16 18:20:10] [INFO] [dapt] MaritimeBERT Loss:   0.5247
[2026-09-16 18:20:10] [INFO] [dapt] Delta Loss (Gain):   +1.0118
[2026-09-16 18:20:10] [INFO] [dapt] Baseline Perplexity: 4.6484
[2026-09-16 18:20:10] [INFO] [dapt] MaritimeBERT PPL:    1.69
[2026-09-16 18:20:10] [INFO] [dapt] Comparison report saved to: /content/drive/MyDrive/Abhishek-TSBC/TSBC-Pipeline/dapt/outputs/experiments/comparison_report.json


## 11. Run Unit Test Suite (Optional Verification)

In [12]:
!pytest dapt/tests/

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0
rootdir: /content/drive/MyDrive/Abhishek-TSBC/TSBC-Pipeline
plugins: anyio-4.14.2, langsmith-0.12.1, typeguard-4.6.0
collected 0 items                                                              

============================ no tests ran in 0.05s =============================
ERROR: file or directory not found: dapt/tests/

